# Network Flow V2 — Step 05: Source-integrity and temporal-readiness gate

**Purpose:** Investigate the mixed values in Edge-IIoTset `frame.time`, assess X-IIoTID timestamp evidence, and document that the current ToN-IoT CSV has no event timestamp.

**Safety:** Read-only access to raw CSV files. No cleaning, split, feature scaling, model fitting or label changes. Results are written only to `reports/audits/` and `docs/`.

**Research decision:** These diagnostics do **not** automatically approve temporal training. A verified timestamp format, observation unit, capture provenance, device grouping, and window-level label rules are still required.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv
import ipaddress
import json
import math
import re
import pandas as pd
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
EDGE = ROOT / "data/raw/edge_iiotset/ML-EdgeIIoT-dataset.csv"
XII = ROOT / "data/raw/x_iiotid/X-IIoTID dataset.csv"
TON = ROOT / "data/raw/ton_iot/ton_iot_train_test_network.csv"
OUTPUT = ROOT / "reports/audits"
DOCS = ROOT / "docs"
OUTPUT.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

for name, path in [('Edge-IIoTset', EDGE), ('X-IIoTID', XII), ('ToN-IoT', TON)]:
    print(f'{name}: {"FOUND" if path.is_file() else "MISSING"} — {path}')
    if not path.is_file():
        raise FileNotFoundError(path)


## 1. Raw Edge-IIoTset CSV shape and `frame.time` patterns

- Inspect **every CSV row** using the standard CSV parser to detect field-count inconsistencies.
- Count time-like, numeric, IP-like and other tokens in `frame.time`.
- Inspect a few original row numbers and associated *attack-type labels*. CSV row numbers include the header on line 1.
- Do **not** silently coerce IP-like strings into timestamps.

A well-formed CSV row can still have semantically misaligned fields, so normal field counts do not prove timestamp correctness.

In [ ]:
def frame_time_kind(raw):
    s = str(raw).strip()
    if not s or s.lower() in {'nan', 'none', 'null', '-', '?'}:
        return 'missing_or_placeholder'
    try:
        ipaddress.IPv4Address(s)
        return 'ipv4_like'
    except ipaddress.AddressValueError:
        pass
    if re.fullmatch(r'[+-]?(?:\d+\.?\d*|\.\d+)', s):
        return 'numeric_like'
    if re.match(r'^\d{4}[-/ ]', s):
        return 'year_prefix_datetime_like'
    return 'other'

edge_kinds = Counter()
edge_by_attack = defaultdict(Counter)
edge_examples = defaultdict(list)
edge_width_errors = []
edge_width = None
edge_n = 0

with EDGE.open('r', encoding='utf-8-sig', newline='', errors='replace') as handle:
    reader = csv.reader(handle)
    header = next(reader)
    edge_width = len(header)
    time_idx = header.index('frame.time')
    attack_idx = header.index('Attack_type')
    for line_number, row in enumerate(reader, start=2):
        edge_n += 1
        if len(row) != edge_width:
            if len(edge_width_errors) < 20:
                edge_width_errors.append({'csv_line': line_number, 'field_count': len(row)})
            continue
        kind = frame_time_kind(row[time_idx])
        edge_kinds[kind] += 1
        edge_by_attack[row[attack_idx]][kind] += 1
        if len(edge_examples[kind]) < 6:
            edge_examples[kind].append({'csv_line': line_number, 'value': row[time_idx],
                                        'Attack_type': row[attack_idx]})

print('Parsed data records:', edge_n, '| expected fields:', edge_width)
print('Field-width mismatches (first 20):', edge_width_errors)
print('frame.time patterns:', dict(edge_kinds))
print('Examples by pattern:')
for k, v in edge_examples.items():
    print(' ', k, v)
print('\nFrame-time representation by attack label (sample of groups):')
edge_by_class = pd.DataFrame.from_dict({k:dict(v) for k,v in edge_by_attack.items()}, orient='index').fillna(0).astype(int)
display(edge_by_class)


## 2. X-IIoTID timestamp plausibility and ordering

X-IIoTID's `Timestamp` looks like Unix seconds, but the CSV may be shuffled. We report ordering reversals **only as diagnostics**; they are not evidence that timestamps are wrong. An epoch-looking value does not establish a complete attack timeline.

We intentionally read a small subset of columns in chunks rather than all 68 columns.

In [ ]:
x_columns = ['Date', 'Timestamp', 'Scr_IP', 'Des_IP', 'class1', 'class2', 'class3']
x_rows = 0
x_epoch_plausible = 0
x_non_numeric = 0
x_missing_dates = 0
x_decreasing_adjacencies = 0
x_equal_adjacencies = 0
x_last_ts = None
x_first_examples = []
x_epoch_by_binary = defaultdict(Counter)
x_date_examples = []
x_unique_seconds = set()
MIN_EPOCH = 1262304000  # 2010-01-01 UTC
MAX_EPOCH = 2082758400  # 2036-01-01 UTC; plausibility range only

for chunk in pd.read_csv(XII, usecols=x_columns, dtype='string', keep_default_na=False,
                         chunksize=30_000, low_memory=False):
    ts = pd.to_numeric(chunk['Timestamp'], errors='coerce')
    values = ts.to_numpy(dtype='float64', na_value=math.nan)
    labels = chunk['class3'].astype(str).to_numpy()
    dates = chunk['Date'].astype(str).to_numpy()
    for i, value in enumerate(values):
        x_rows += 1
        raw_date = dates[i].strip()
        if not raw_date or raw_date.lower() in {'nan','none','null','-','?'}:
            x_missing_dates += 1
        elif len(x_date_examples) < 8 and raw_date not in x_date_examples:
            x_date_examples.append(raw_date)
        if math.isfinite(value) and MIN_EPOCH <= value < MAX_EPOCH:
            x_epoch_plausible += 1
            x_epoch_by_binary[labels[i]]['plausible'] += 1
            current = int(value)
            x_unique_seconds.add(current)
            if x_last_ts is not None:
                if current < x_last_ts:
                    x_decreasing_adjacencies += 1
                elif current == x_last_ts:
                    x_equal_adjacencies += 1
            x_last_ts = current
            if len(x_first_examples) < 10:
                x_first_examples.append({'date':raw_date,'timestamp':current,
                    'utc':datetime.fromtimestamp(current, tz=timezone.utc).isoformat(),
                    'class3':labels[i]})
        else:
            x_non_numeric += 1
            x_epoch_by_binary[labels[i]]['invalid_or_nonplausible'] += 1

x_summary = {'records': x_rows,'plausible_epoch_seconds':x_epoch_plausible,
    'invalid_or_nonplausible':x_non_numeric, 'date_missing_or_placeholder':x_missing_dates,
    'unique_plausible_epoch_seconds':len(x_unique_seconds),
    'decreasing_adjacent_valid_timestamps':x_decreasing_adjacencies,
    'equal_adjacent_valid_timestamps':x_equal_adjacencies,
    'per_binary_label':{k:dict(v) for k,v in x_epoch_by_binary.items()},
    'date_examples':x_date_examples,'timestamp_examples':x_first_examples}
print(json.dumps(x_summary, indent=2))


## 3. ToN-IoT time availability and dataset plan

We check the header only. Do not confuse `duration` (length of one connection) with a timestamp identifying **when** that connection occurred.

In [ ]:
with TON.open('r', encoding='utf-8-sig', newline='', errors='replace') as handle:
    ton_columns = next(csv.reader(handle))

possible_time_columns = [c for c in ton_columns if c.lower().strip() in
                         {'timestamp','time','ts','frame.time','date','start_time','event_time'}]
print('ToN-IoT explicit timestamp candidates:', possible_time_columns)
print('duration present:', 'duration' in ton_columns)
print('Important: duration != event timestamp. Temporal windows are not approved for this CSV.')


## 4. Save an evidence-based decision note

`REVIEW_REQUIRED` does not imply dataset is unusable for static classification; it only means the **temporal-window claim** lacks adequate evidence.

This file does not change any CSV or create training/validation/test splits.

In [ ]:
report = {
  'status': 'READ_ONLY_PRE_SPLIT_DIAGNOSTICS',
  'edge_iiotset': {
    'records':edge_n,'expected_columns':edge_width,
    'field_count_errors_first_20':edge_width_errors,
    'frame_time_kinds':dict(edge_kinds),
    'frame_time_by_attack_type':{k:dict(v) for k,v in edge_by_attack.items()},
    'temporal_ready':'REVIEW_REQUIRED',
    'note':'IP-like/numeric/year-prefix mixed tokens; verify source export and original capture semantics'
  },
  'x_iiotid':x_summary | {'temporal_ready':'REVIEW_REQUIRED',
    'note':'Plausible epoch seconds do not verify capture grouping, time alignment or window label truth'},
  'ton_iot':{'explicit_timestamp_columns':possible_time_columns,
    'has_duration':('duration' in ton_columns), 'temporal_ready':False,
    'note':'No event timestamp in current downloaded CSV'},
  'next_step':'Investigate Edge frame.time values against original release, verify X capture/time/label provenance, then choose grouped split; static baselines may proceed independently with leakage-aware feature selection.'
}
json_path = OUTPUT / 'three_dataset_temporal_source_gate_v1.json'
json_path.write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
md_path = DOCS / 'THREE_DATASET_TEMPORAL_SOURCE_GATE_V1.md'
lines = [
    '# Three-Dataset Temporal and Source Integrity Gate — V1', '',
    'Dataset operations: **read-only**; training and split generation: **not performed**.', '',
    '## Edge-IIoTset `frame.time` representation', '',
    '| Pattern | Records |', '|---|---:|',
]
lines += [f'| {kind} | {count:,} |' for kind, count in sorted(edge_kinds.items())]
lines += ['',f'CSV records inspected: **{edge_n:,}**. Field-count issue examples: **{len(edge_width_errors)}** (limited to first 20).',
          '', 'Temporal readiness: **REVIEW REQUIRED** — confirm original export semantics, capture ordering, device identity and window-level labels.',
          '', '## X-IIoTID timestamp evidence', '',
          f'- Rows examined: **{x_rows:,}**',
          f'- Plausible epoch seconds: **{x_epoch_plausible:,}**',
          f'- Invalid/nonplausible timestamps: **{x_non_numeric:,}**',
          f'- Distinct candidate seconds: **{len(x_unique_seconds):,}**',
          f'- Adjacent descending timestamps (raw file order): **{x_decreasing_adjacencies:,}**',
          '- Temporal readiness: **REVIEW REQUIRED** — corroborate capture/session and timestamp-to-label relationships.',
          '', '## ToN-IoT', '',
          '- The downloaded CSV does not provide a verified event timestamp.',
          '- `duration` is a duration, not a window start time.',
          '- 5/10/15/20-second aggregation is **not supported** by this CSV as-is.',
          '', '## Next decision', '',
          'Run dataset-specific static intrusion-classification baselines with group-/capture-aware partitions where possible; do not claim independent temporal/cross-dataset detection until provenance and feature meanings are verified.',
          '']
md_path.write_text('\n'.join(lines),encoding='utf-8')
print('Report:',json_path)
print('Decision note:',md_path)
print('Raw data unchanged. No training/splitting performed.')
